In [ ]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import numpy as np
from torchvision import transforms
from ultralytics import YOLO
from IPython.display import Image, display
from ultralytics import RTDETR
import gc; import torch

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
zip_path = '/content/drive/MyDrive/yolo_dataset.zip'
extract_path = '/content/yolo_dataset'

!unzip -q -o "{zip_path}" -d "{extract_path}"

In [ ]:
#경로설정
train_img_path = os.path.join(extract_path, 'train', 'images')
train_label_path = os.path.join(extract_path, 'train', 'labels')
val_img_path = os.path.join(extract_path, 'val', 'images')
val_label_path = os.path.join(extract_path, 'val', 'labels')
test_img_path = os.path.join(extract_path, 'test', 'images')

paths = {
    "Train 이미지": train_img_path,
    "Train 라벨": train_label_path,
    "Val 이미지": val_img_path,
    "Val 라벨": val_label_path,
    "Test 이미지": test_img_path
}

for name, path in paths.items():
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"{name}: {count}개")
    else:
        print(f"{name}: 경로 없음")

# 최종 합계 (이미지 기준)
total_images = len(os.listdir(train_img_path)) + len(os.listdir(val_img_path))
print(f"총 이미지 합계: {total_images}개")

Train 이미지: 232개
Train 라벨: 232개
Val 이미지: 118개
Val 라벨: 118개
Test 이미지: 842개
총 이미지 합계: 350개


In [ ]:
#디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
model = YOLO('yolo12n.pt')

#학습 시작
model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=30,
    imgsz=1280,
    batch=8,
    device=device,
    workers=2,
    name='pill_v12_project'
)

engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pill_v12_project, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=N

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  6,  7,  9, 11, 12, 13, 14, 15, 16, 17, 18, 21, 22, 23, 24, 25, 27, 28, 29, 30, 31, 32, 39, 41, 42, 43, 45, 48, 50, 51, 55])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7baa16de0920>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.0380

In [ ]:
best_model = YOLO('/content/runs/detect/pill_v12_project/weights/best.pt')
metrics = best_model.val(split='val')

all_maps = metrics.box.maps

score_list = all_maps[5:]
mAP = np.mean(score_list)

print(f"mAP@[0.75:0.95]: {mAP:.4f}")

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv12n summary (fused): 159 layers, 2,567,648 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3598.3±1417.9 MB/s, size: 1667.4 KB)
val: Scanning /content/yolo_dataset/val/labels.cache... 118 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 118/118 30.9Mit/s 0.0s
val: /content/yolo_dataset/val/images/K-003351-016262-018357_0_2_0_2_75_000_200.png: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     6.8878]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.3s/it 10.8s
                   all        117        117      0.529      0.383      0.208      0.205
           가바토파정 100mg          3          3          1          0      0.103      0.103
             글리아타민연질캡슐          2          2          0          0     0.0255      0.024
        글리틴정(콜린알포세레이트)          3          3 

In [ ]:
test_img_dir = '/content/yolo_dataset/test/images'

if not os.path.exists(test_img_dir):
    test_img_dir = '/content/yolo_dataset/yolo_dataset/test/images'

all_test_images = [os.path.join(test_img_dir, f) for f in os.listdir(test_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
sample_images = all_test_images[:10]

results = best_model.predict(
    source=sample_images,
    save=True,
    conf=0.1,
    iou=0.45
)

save_path = results[0].save_dir

for res in results:
    fname = os.path.basename(res.path)
    result_img_path = os.path.join(save_path, fname)

    print(f"검출 결과: {fname}")
    display(Image(filename=result_img_path, width=600))

Output hidden; open in https://colab.research.google.com to view.

YOLOv12 batch = 8 epoch=30 기준 15분, mAP[0.75:0/95] = 0.2068


* 작고 큰 몇몇 알약들을 검출을 못함, 검출을 하더라도 신뢰도 점수 매우 낮음

RT-DETR 테스트

In [ ]:
del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
model_rt = RTDETR('rtdetr-l.pt')

#학습 시작
model_rt.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=30,
    imgsz=1280,
    batch=4,
    device=device,
    workers=2,
    name='pill_rtdetr_project'
)

engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pill_rtdetr_project, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, proje

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      10.9G      0.859      2.682     0.6059         10       1280: 100% ━━━━━━━━━━━━ 58/58 1.2s/it 1:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.2it/s 12.1s
                   all        117        117    0.00946     0.0549    0.00913    0.00707

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      11.2G     0.3767      3.253     0.2332          6       1280: 100% ━━━━━━━━━━━━ 58/58 1.1s/it 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.9it/s 8.0s
                   all        117        117      0.615     0.0177     0.0046    0.00439

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      11.2G    0.07667       1.16    0.04396          4       1280: 100% ━━━━━━━━━━━━ 58/58 1.1s/it 1:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.9it/s 7.8s
                   all        117        117      0.343      0.549      0.284      0.274

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      11.1G     0.0682       1.16    0.04249          4       1280: 100% ━━━━━━━━━━━━ 58/58 1.1s/it 1:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.9it/s 8.1s
                   all        117        117      0.344      0.545      0.295      0.286

30 epochs completed in 0.639 hours.
Optimizer stripped from /content/runs/detect/pill_rtdetr_project/weights/last.pt, 66.7MB
Optimizer stripped from /content/runs/detect/pill_rtdetr_project/weights/best.pt, 66.7MB

Validating /content/runs/detect/pill_rtdetr_project/weights/best.pt...
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 32,098,820 parameters, 0 gradients, 103.7 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.5it/s 10.1s
                   all        117        117      0.344      0.549      0.314      0.303
 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  6,  7,  9, 11, 12, 13, 14, 15, 16, 17, 18, 21, 22, 23, 24, 25, 27, 28, 29, 30, 31, 32, 39, 41, 42, 43, 45, 48, 50, 51, 55])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ba993c1f470>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.0380

RT-DETR은 Transformer 구조를 채택하여 이미지 전체적인 맥락을 파악하는 데 유리

In [ ]:
best_model_rt = YOLO('/content/runs/detect/pill_rtdetr_project/weights/best.pt')
metrics_rt = best_model_rt.val(split='val')

all_maps_rt = metrics_rt.box.maps

score_list_rt = all_maps_rt[5:]
mAP_rt = np.mean(score_list_rt)

print(f"mAP@[0.75:0.95]: {mAP_rt:.4f}")

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 32,098,820 parameters, 0 gradients, 103.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5070.9±1147.4 MB/s, size: 1662.7 KB)
val: Scanning /content/yolo_dataset/val/labels.cache... 118 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 118/118 33.0Mit/s 0.0s
val: /content/yolo_dataset/val/images/K-003351-016262-018357_0_2_0_2_75_000_200.png: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     6.8878]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.8s/it 22.4s
                   all        117        117      0.311      0.549      0.312      0.302
           가바토파정 100mg          3          3      0.324          1      0.583      0.582
             글리아타민연질캡슐          2          2      0.472          1      0.663      0.619
        글리틴정(콜린알포세레이트)          3          3     

In [ ]:
test_img_dir = '/content/yolo_dataset/test/images'

if not os.path.exists(test_img_dir):
    test_img_dir = '/content/yolo_dataset/yolo_dataset/test/images'

all_test_images = [os.path.join(test_img_dir, f) for f in os.listdir(test_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
sample_images = all_test_images[:10]

results = best_model_rt.predict(
    source=sample_images,
    save=True,
    conf=0.1,
    iou=0.45
)

save_path = results[0].save_dir

for res in results:
    fname = os.path.basename(res.path)
    result_img_path = os.path.join(save_path, fname)

    print(f"검출 결과: {fname}")
    display(Image(filename=result_img_path, width=600))

Output hidden; open in https://colab.research.google.com to view.

RT-DETR batch = 4 epoch=30 기준 38분, mAP[0.75:0/95] = 0.2817
